# 05 · Evaluate the doctor models and baselines

**SimClinic, CS425 Group 7 (doctor model).** Every doctor is tested on the same held-out test cards.

**Doctors compared**
| Doctor | What it is |
|---|---|
| NB doctor | Asks the symptom with the highest information gain under Symcat naive Bayes, then diagnoses with naive Bayes (the policy behind the earlier stand-in training data) |
| GPT-2 pretrained | Base GPT-2 with a plain-text prompt only (no fine-tuning); diagnosis ranked by likelihood |
| GPT-2 small / medium (fine-tuned) | Best run of each size from notebook 04 |
| LLaMA-3-8B / Mistral-7B × zero-shot, few-shot, CoT | Instruction-tuned, prompted only (weights untouched), run locally through Ollama |

**Two tests**
1. **Auto mode.** The doctor interviews the rule-based card patient, so it has to ask the right questions *and* reason. We report top-1/top-3 accuracy against the hidden disease, questions asked, and the share of card symptoms found. We also report **naive Bayes on the symptoms that same dialogue revealed**: the gap between a doctor's accuracy and NB on its own evidence separates how well it *asks* from how well it *reasons*.
2. **Fixed transcripts.** Every doctor diagnoses from the same held-out dialogues, which tests reasoning alone.

All records are cached in `data/processed/doctor/eval/`, so a long LLM run that is interrupted resumes where it stopped.

**Caveat.** Auto mode still uses the rule-based card patient (`simclinic/patients.py`). It understands questions only through the symptom lexicon and a few history-taking patterns, and it answers in short template sentences. The fine-tuned GPT-2 learned from the generated consultations of notebook 07, where the patients talk naturally, so in auto mode it meets a patient unlike the ones it trained on. Sections 5.2 and 5.4 show where that patient misreads a doctor's questions. Absolute numbers will change once the GPT-2 patient model replaces it. The fixed transcripts are held-out generated consultations that no doctor has seen.

## 0. Config

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

MAX_QUESTIONS = 10
N_TEST = None                  # None = every test card
RUN_LLMS = True                # skipped automatically if the backend is unreachable
LLM_BACKEND = "ollama"         # "ollama" (local, quantised) or "hf" (gated HF checkpoints, 4-bit)
LLM_MODELS = ["llama3-8b", "mistral-7b"]
STRATEGIES = ["zero_shot", "few_shot", "cot"]
LLM_CASES_PER_DISEASE = 1      # 1 = 61 cases per LLM configuration; None = every test card (several hours)
FEW_SHOT_EXAMPLES = 2

DATA_DIR = ROOT / "data" / "processed" / "doctor"
CKPT_DIR = ROOT / "checkpoints" / "doctor"
OUT_DIR = DATA_DIR / "eval"
SEED = 42

## 1. Load cards, dialogues and the Symcat model

In [2]:
import gc
import json
import re
import time
import urllib.request
from collections import Counter
import pandas as pd
import torch
from transformers import set_seed
from simclinic import cards as C
from simclinic.consult import diagnose_transcript, run_consultation, score, summarise
from simclinic.doctors import GPT2Doctor, LLMDoctor, NBDoctor
from simclinic.lexicon import Lexicon, revealed_symptoms
from simclinic.llm import MODELS, HFChat, OllamaChat
from simclinic.naive_bayes import SymcatNaiveBayes
from simclinic.patients import CardPatient
from simclinic.prompts import DiseaseMatcher, parse_reply
from simclinic.symcat import Condition

pd.set_option("display.max_colwidth", 140)

def read_jsonl(path):
    path = Path(path)
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()] if path.exists() else []

CONDITIONS = json.loads((DATA_DIR / "conditions.json").read_text())
symcat_table = {n: Condition(**c) for n, c in json.loads((DATA_DIR / "symcat_conditions.json").read_text()).items()}
symptom_vocab = sorted({s for n in CONDITIONS for s in symcat_table[n].symptoms})
lexicon = Lexicon.load(symptoms=symptom_vocab)
nb = SymcatNaiveBayes({n: symcat_table[n].symptoms for n in CONDITIONS})
matcher = DiseaseMatcher({n: C.disease_names(symcat_table[n]) for n in CONDITIONS})

test_cards = read_jsonl(DATA_DIR / "cards" / "test.jsonl")[:N_TEST]
test_dialogues = read_jsonl(DATA_DIR / "dialogues" / "test.jsonl")
train_dialogues = read_jsonl(DATA_DIR / "dialogues" / "train.jsonl")
card_by_id = {c["card_id"]: c for c in read_jsonl(DATA_DIR / "cards" / "test.jsonl")}
seed_of = {c["card_id"]: SEED + i for i, c in enumerate(test_cards)}    # same patient randomness for every doctor

def stratified(cases, per_disease, key):
    if per_disease is None:
        return cases
    taken = Counter()
    out = []
    for c in cases:
        if taken[key(c)] < per_disease:
            taken[key(c)] += 1
            out.append(c)
    return out

llm_cards = stratified(test_cards, LLM_CASES_PER_DISEASE, key=lambda c: c["disease"])
llm_dialogues = stratified(test_dialogues, LLM_CASES_PER_DISEASE, key=lambda d: d["disease"])
print(f"{len(test_cards)} test cards ({len({c['disease'] for c in test_cards})} diseases), "
      f"{len(test_dialogues)} test dialogues; LLM subset: {len(llm_cards)} cards / {len(llm_dialogues)} dialogues")

201 test cards (61 diseases), 201 test dialogues; LLM subset: 61 cards / 61 dialogues


## 2. Doctors
The fine-tuned GPT-2 models are the best small and the best medium run from the notebook 04 grid. The few-shot prompt uses two training dialogues of different diseases. The CoT prompt stays zero-shot, so the effects of examples and of reasoning can be measured separately.

In [3]:
grid = pd.read_csv(CKPT_DIR / "grid_results.csv")
best_per_size = (grid.sort_values(["dev_top3", "dev_top1", "dev_loss"], ascending=[False, False, True])
                     .groupby("model", sort=False).head(1).set_index("model"))

def pick_examples(k):
    pool = [d for d in train_dialogues if 4 <= sum(t["role"] == "doctor" for t in d["turns"]) - 1 <= 6]
    out, used = [], set()
    for d in sorted(pool, key=lambda d: d["dialogue_id"]):
        if d["disease"] not in used and len(out) < k:
            out.append(d)
            used.add(d["disease"])
    return out

few_shot_examples = pick_examples(FEW_SHOT_EXAMPLES)

def llm_backend_ready():
    if not RUN_LLMS:
        return False
    if LLM_BACKEND == "hf":
        return True
    try:
        tags = json.loads(urllib.request.urlopen("http://localhost:11434/api/tags", timeout=5).read())
        have = {m["name"] for m in tags["models"]}
        missing = [MODELS[m]["ollama"] for m in LLM_MODELS if MODELS[m]["ollama"] not in have]
        if missing:
            print("missing Ollama models (run `ollama pull <tag>`):", missing)
        return not missing
    except OSError as e:
        print("Ollama server not reachable, skipping LLM baselines:", e)
        return False

_chats = {}
def chat_for(model_key):
    if model_key not in _chats:
        _chats[model_key] = (OllamaChat(MODELS[model_key]["ollama"]) if LLM_BACKEND == "ollama"
                             else HFChat(MODELS[model_key]["hf"]))
    return _chats[model_key]

def free_llms():
    """Unload the LLMs before a GPT-2 doctor runs: Ollama keeps the last model on the GPU for minutes after
    its last request (even from an earlier session), and GPT-2 then runs out of memory."""
    if LLM_BACKEND == "ollama" and LLM_DOCTORS:
        for m in LLM_MODELS:
            OllamaChat(MODELS[m]["ollama"]).unload()
    _chats.clear()
    gc.collect()
    torch.cuda.empty_cache()

DOCTORS = {
    "NB doctor": lambda: NBDoctor(nb, lexicon, max_questions=MAX_QUESTIONS, seed=SEED),
    "GPT-2 pretrained (prompt only)": lambda: GPT2Doctor.from_pretrained("gpt2", CONDITIONS, max_new_tokens=40),
}
for size, label in [("gpt2", "GPT-2 small fine-tuned"), ("gpt2-medium", "GPT-2 medium fine-tuned")]:
    if size in best_per_size.index:
        DOCTORS[label] = (lambda p=best_per_size.loc[size, "path"]: GPT2Doctor.from_pretrained(str(ROOT / p), CONDITIONS))
LLM_DOCTORS = {}
if llm_backend_ready():
    for m in LLM_MODELS:
        for s in STRATEGIES:
            LLM_DOCTORS[f"{m} {s}"] = (lambda m=m, s=s: LLMDoctor(chat_for(m), CONDITIONS, matcher, strategy=s,
                                                                  examples=few_shot_examples,
                                                                  max_questions=MAX_QUESTIONS))
print("doctors:", list(DOCTORS) + list(LLM_DOCTORS))

doctors: ['NB doctor', 'GPT-2 pretrained (prompt only)', 'GPT-2 small fine-tuned', 'GPT-2 medium fine-tuned', 'llama3-8b zero_shot', 'llama3-8b few_shot', 'llama3-8b cot', 'mistral-7b zero_shot', 'mistral-7b few_shot', 'mistral-7b cot']


## 3. Auto mode

In [4]:
def slug(name):
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")

def release(doctor):
    """Free a GPT-2 doctor's GPU memory; otherwise PyTorch keeps it cached and Ollama has to put the 7-8B
    models partly on the CPU, which is several times slower."""
    del doctor
    gc.collect()
    torch.cuda.empty_cache()

def run_auto(name, make_doctor, cases):
    path = OUT_DIR / "auto" / f"{slug(name)}.jsonl"
    path.parent.mkdir(parents=True, exist_ok=True)
    done = {r["card_id"] for r in read_jsonl(path)}
    todo = [c for c in cases if c["card_id"] not in done]
    if todo:
        doctor, t0 = make_doctor(), time.time()
        with open(path, "a", encoding="utf-8") as f:
            for i, card in enumerate(todo):
                set_seed(seed_of[card["card_id"]])
                start = time.time()
                rec = run_consultation(doctor, CardPatient(card, lexicon, seed=seed_of[card["card_id"]]),
                                       max_questions=MAX_QUESTIONS)
                row = {"doctor": name, "card_id": card["card_id"], "disease": card["disease"],
                       **score(rec, card, nb, lexicon), "seconds": time.time() - start, **rec}
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                f.flush()
        print(f"{name}: {len(todo)} cases in {(time.time() - t0) / 60:.1f} min")
        release(doctor)
    wanted = {c["card_id"] for c in cases}
    return [r for r in read_jsonl(path) if r["card_id"] in wanted]

auto_rows = []
free_llms()
for name, make in DOCTORS.items():
    auto_rows += run_auto(name, make, test_cards)
for name, make in LLM_DOCTORS.items():
    auto_rows += run_auto(name, make, llm_cards)

### 3.1 Results on every test card (local doctors)

In [5]:
auto_df = pd.DataFrame(auto_rows)
summarise([r for r in auto_rows if r["doctor"] in DOCTORS])

,cases,top-1,top-3,NB top-1 (same evidence),NB top-3 (same evidence),questions,card symptoms found,forced to diagnose,no valid diagnosis
doctor,,,,,,,,,
NB doctor,201,0.706,0.940,0.706,0.940,7.463,0.715,0.423,0.0
GPT-2 pretrained (prompt only),201,0.050,0.080,0.572,0.846,10.000,0.422,1.000,0.0
GPT-2 small fine-tuned,201,0.687,0.861,0.542,0.861,6.174,0.580,0.000,0.0
GPT-2 medium fine-tuned,201,0.726,0.900,0.592,0.851,6.353,0.616,0.000,0.0


### 3.2 Results on the LLM subset (one card per disease), every doctor
The same cards for every row, so the rows can be compared directly.

In [6]:
llm_ids = {c["card_id"] for c in llm_cards}
auto_subset = summarise([r for r in auto_rows if r["card_id"] in llm_ids])
auto_subset

,cases,top-1,top-3,NB top-1 (same evidence),NB top-3 (same evidence),questions,card symptoms found,forced to diagnose,no valid diagnosis
doctor,,,,,,,,,
NB doctor,61,0.738,0.951,0.738,0.951,7.361,0.713,0.361,0.000
GPT-2 pretrained (prompt only),61,0.033,0.082,0.590,0.836,10.000,0.445,1.000,0.000
GPT-2 small fine-tuned,61,0.672,0.885,0.557,0.869,6.082,0.580,0.000,0.000
GPT-2 medium fine-tuned,61,0.770,0.918,0.623,0.902,6.246,0.631,0.000,0.000
llama3-8b zero_shot,61,0.377,0.639,0.541,0.803,3.262,0.483,0.000,0.000
llama3-8b few_shot,61,0.295,0.574,0.525,0.836,5.508,0.572,0.000,0.000
llama3-8b cot,61,0.328,0.557,0.574,0.852,6.607,0.584,0.000,0.000
mistral-7b zero_shot,61,0.295,0.492,0.607,0.852,8.492,0.616,0.311,0.000
mistral-7b few_shot,61,0.361,0.623,0.590,0.869,9.230,0.597,0.541,0.000


## 4. Fixed transcripts (reasoning only)
Each doctor diagnoses from the same held-out generated consultation (notebook 07), with the closing remark removed. Here NB is simply naive Bayes on the symptoms the transcript reveals.

In [7]:
def run_transcripts(name, make_doctor, dialogues):
    path = OUT_DIR / "transcripts" / f"{slug(name)}.jsonl"
    path.parent.mkdir(parents=True, exist_ok=True)
    done = {r["dialogue_id"] for r in read_jsonl(path)}
    todo = [d for d in dialogues if d["dialogue_id"] not in done]
    if todo:
        doctor = make_doctor()
        with open(path, "a", encoding="utf-8") as f:
            for d in todo:
                set_seed(SEED)
                dx = diagnose_transcript(doctor, d["turns"])
                f.write(json.dumps({"doctor": name, "dialogue_id": d["dialogue_id"], "disease": d["disease"],
                                    "diagnosis": dx, "top1": dx[:1] == [d["disease"]],
                                    "top3": d["disease"] in dx[:3]}) + "\n")
        release(doctor)
    wanted = {d["dialogue_id"] for d in dialogues}
    return [r for r in read_jsonl(path) if r["dialogue_id"] in wanted]

transcript_doctors = {"Naive Bayes": None, **{k: v for k, v in DOCTORS.items() if k != "NB doctor"}}
free_llms()
transcript_rows = []
for d in test_dialogues:
    dx = [x for x, _ in nb.rank(revealed_symptoms(d["turns"][:-1], lexicon), k=3)]
    transcript_rows.append({"doctor": "Naive Bayes", "dialogue_id": d["dialogue_id"], "disease": d["disease"],
                            "diagnosis": dx, "top1": dx[:1] == [d["disease"]], "top3": d["disease"] in dx})
for name, make in transcript_doctors.items():
    if make is not None:
        transcript_rows += run_transcripts(name, make, test_dialogues)
for name, make in LLM_DOCTORS.items():
    transcript_rows += run_transcripts(name, make, llm_dialogues)

tr = pd.DataFrame(transcript_rows)
llm_dialogue_ids = {d["dialogue_id"] for d in llm_dialogues}
transcript_summary = pd.concat({
    "all test dialogues": tr.groupby("doctor", sort=False)[["top1", "top3"]].mean(),
    "LLM subset": tr[tr["dialogue_id"].isin(llm_dialogue_ids)].groupby("doctor", sort=False)[["top1", "top3"]].mean(),
}, axis=1).round(3)
transcript_summary.loc[[d for d in transcript_summary.index if d not in DOCTORS and d != "Naive Bayes"],
                       "all test dialogues"] = float("nan")       # LLMs only ran on the subset
transcript_summary

all test dialogues        LLM subset       
                                             top1   top3       top1   top3
doctor                                                                    
Naive Bayes                                 0.642  0.910      0.738  0.934
GPT-2 pretrained (prompt only)              0.050  0.090      0.049  0.098
GPT-2 small fine-tuned                      0.816  0.990      0.803  1.000
GPT-2 medium fine-tuned                     0.856  0.975      0.902  1.000
llama3-8b zero_shot                           NaN    NaN      0.410  0.672
llama3-8b few_shot                            NaN    NaN      0.361  0.689
llama3-8b cot                                 NaN    NaN      0.328  0.541
mistral-7b zero_shot                          NaN    NaN      0.361  0.639
mistral-7b few_shot                           NaN    NaN      0.393  0.623
mistral-7b cot                                NaN    NaN      0.377  0.574

## 5. Error analysis
### 5.1 Most frequent confusions (auto mode, top-1)

In [8]:
wrong = auto_df[~auto_df["top1"] & auto_df["diagnosis"].map(bool)]
conf = (wrong.assign(predicted=wrong["diagnosis"].str[0])
             .groupby(["doctor", "disease", "predicted"]).size().rename("count").reset_index()
             .sort_values(["doctor", "count"], ascending=[True, False]).groupby("doctor").head(3))
conf

,doctor,disease,predicted,count
17,GPT-2 medium fine-tuned,Fungal infection of the skin,Chickenpox,4
5,GPT-2 medium fine-tuned,Benign paroxysmal positional vertigo (BPPV),Heat exhaustion,3
11,GPT-2 medium fine-tuned,Contact dermatitis,Chickenpox,3
43,GPT-2 pretrained (prompt only),Aphthous ulcer,Pneumonia,4
47,GPT-2 pretrained (prompt only),Asthma,Pneumonia,4
60,GPT-2 pretrained (prompt only),Common cold,Pneumonia,4
142,GPT-2 small fine-tuned,Fungal infection of the skin,Chickenpox,4
138,GPT-2 small fine-tuned,Contact dermatitis,Chickenpox,3
164,GPT-2 small fine-tuned,Strep throat,Mononucleosis,3
169,NB doctor,Acute sinusitis,Seasonal allergies (hay fever),3


### 5.2 How the rule-based patient copes with each doctor's questions
"I'm not sure, doctor." is the patient's fallback when a question matches neither the lexicon nor a history-taking pattern. A high rate means that doctor's evidence is under-counted, which is a limitation of the rule-based patient rather than of the doctor. The rate misses a second failure: a question that contains a lexicon phrase but asks something else gets a yes/no about that symptom ("Where exactly is the pain in your ear?" → "Yes, I've had ear pain.", section 5.4).

In [9]:
def patient_fallback_rate(rows):
    turns = [t for r in rows for t in r["turns"] if t["role"] == "patient"]
    return sum(t["text"] == "I'm not sure, doctor." for t in turns) / max(1, len(turns))

pd.DataFrame({name: {"patient 'not sure' rate": patient_fallback_rate(g.to_dict("records")),
                     "repeated questions per case": g["turns"].map(
                         lambda ts: (lambda q: len(q) - len(set(q)))([t["text"] for t in ts if t["role"] == "doctor"])).mean()}
              for name, g in auto_df.groupby("doctor", sort=False)}).T.round(3)

,patient 'not sure' rate,repeated questions per case
NB doctor,0.000,0.000
GPT-2 pretrained (prompt only),0.684,4.547
GPT-2 small fine-tuned,0.004,0.000
GPT-2 medium fine-tuned,0.003,0.000
llama3-8b zero_shot,0.005,0.033
llama3-8b few_shot,0.003,0.131
llama3-8b cot,0.002,0.033
mistral-7b zero_shot,0.002,0.213
mistral-7b few_shot,0.007,0.279
mistral-7b cot,0.002,0.000


### 5.3 LLM output problems
Replies that gave no usable diagnosis, or listed a condition outside the candidate list.

In [10]:
if LLM_DOCTORS:
    issues = []
    for r in auto_rows:
        if r["doctor"] in LLM_DOCTORS:
            parsed = parse_reply(r["raw"] or "", matcher)
            issues.append({"doctor": r["doctor"], "no diagnosis": not r["diagnosis"],
                           "fewer than 3 valid names": len(r["diagnosis"]) < 3,
                           "unmatched names": "; ".join(parsed.unmatched or [])})
    issues = pd.DataFrame(issues)
    display(issues.groupby("doctor", sort=False)[["no diagnosis", "fewer than 3 valid names"]].mean().round(3))
    display(issues[issues["unmatched names"] != ""]["unmatched names"].value_counts().head(10).to_frame("count"))

,no diagnosis,fewer than 3 valid names
doctor,,
llama3-8b zero_shot,0.000,0.180
llama3-8b few_shot,0.000,0.082
llama3-8b cot,0.000,0.541
mistral-7b zero_shot,0.000,0.197
mistral-7b few_shot,0.000,0.344
mistral-7b cot,0.016,0.525


,count
unmatched names,
Allergies,9
None; None,6
Pharyngitis,4
Allergic reaction,3
Acute pharyngitis,2
Labyrinthitis,2
Second most likely: Flu,2
Sleepwalking,1
DIAGNOSIS: Flu,1


### 5.4 Example consultations

In [11]:
def show(name, k=0):
    rows = [r for r in auto_rows if r["doctor"] == name and r["card_id"] in llm_ids]
    if not rows:
        return
    r = rows[k]
    card = card_by_id[r["card_id"]]
    print(f"=== {name} | hidden: {card['disease']} | card: {', '.join(card['symptoms'])}")
    for t in r["turns"]:
        print(f"  {t['role']:>7}: {t['text']}")
    if r.get("reasoning"):
        print("  [reasoning]", r["reasoning"].replace("\n", "\n               "))
    print(f"  top-3: {'; '.join(r['diagnosis'])}  ({'correct' if r['top1'] else 'in top-3' if r['top3'] else 'wrong'})\n")

for name in ["GPT-2 medium fine-tuned", "GPT-2 small fine-tuned", "llama3-8b cot", "mistral-7b few_shot"]:
    show(name, k=5)

=== GPT-2 medium fine-tuned | hidden: Otitis externa (swimmer's ear) | card: Ear pain, Fluid in ear, Redness in ear, Bleeding from ear
   doctor: What brings you in today?
  patient: Hi doctor. So it started this morning. I've had ear pain and it's not getting better. I was hoping it would go away on its own.
   doctor: Where exactly is the pain in your ear?
  patient: Yes, I've had ear pain. I was hoping it would go away on its own.
   doctor: Does the pain spread anywhere else?
  patient: It's hard to describe, it just feels uncomfortable.
   doctor: Do you have any other symptoms?
  patient: Now that you mention it, I've also had fluid coming out of the ear.
   doctor: Does your ear feel red or swollen?
  patient: Yes, I've had a red ear.
   doctor: Have you noticed any changes in your hearing?
  patient: Nope.
   doctor: Is your ear red or swollen?
  patient: Yeah, I have a red ear.
   doctor: Thank you for letting me know about all of that.
  top-3: Otitis externa (swimmer's ear);

## 6. Save summaries

In [12]:
summarise([r for r in auto_rows if r["doctor"] in DOCTORS]).to_csv(OUT_DIR / "auto_summary_all_test_cards.csv")
auto_subset.to_csv(OUT_DIR / "auto_summary_llm_subset.csv")
transcript_summary.to_csv(OUT_DIR / "transcript_summary.csv")
free_llms()                       # give the GPU and RAM back
print("saved ->", OUT_DIR)

saved -> /home/kth/Documents/Github/project/model/data/processed/doctor/eval


## 7. Findings
From the run of 2026-10-28: generated data from notebook 07, best run of each size from notebook 04.

**Auto mode (201 test cards).**
- **Best top-1:** fine-tuned GPT-2 medium (0.726), ahead of the NB doctor (0.706) and GPT-2 small (0.687).
- **Best top-3:** the NB doctor (0.940 vs 0.900 for medium). It also finds more card symptoms (0.72 vs 0.62), but it asks more questions (7.5 vs 6.4) and hits the 10-question limit in 42% of cases.
- **Pretrained GPT-2:** near zero (0.05). It repeats questions (4.5 per case), and the patient answers "not sure" to 68% of them.

**Asking vs reasoning.** The two kinds of doctor fail in opposite places.
- **Fine-tuned GPT-2 reasons well.** On GPT-2 medium's dialogues, naive Bayes run on the same evidence would score 0.59 top-1, while the model scores 0.73. Naive Bayes sees only the symptoms the lexicon finds; the GPT-2 doctor also sees onset and the patient's wording.
- **LLMs ask well but reason poorly.** Naive Bayes on the evidence they collect scores 0.53–0.66, while the LLMs themselves score 0.30–0.38. Their evidence is reasonable; the step from it to one of 61 candidates is where they fail.

**LLM baselines (61 cases, one per disease).**
- **Accuracy:** all six setups score 0.30–0.38 top-1 and 0.49–0.64 top-3, against 0.770 for GPT-2 medium and 0.738 for the NB doctor on the same cases.
- **Prompting:** neither few-shot nor CoT helps consistently. LLaMA-3 does best zero-shot (0.377) and Mistral does best few-shot (0.361).
- **CoT output:** about half of CoT replies list fewer than three valid candidate names, often naming conditions outside the list ("Allergies", "Pharyngitis").
- **Interview length:** LLaMA-3 zero-shot stops after 3.3 questions on average. Mistral runs to the 10-question limit in 18–54% of cases.

**Fixed transcripts.**
- **Fine-tuned GPT-2:** top-1 0.856 (medium) and 0.816 (small), with top-3 at least 0.975.
- **Baselines:** 0.642 for naive Bayes on the symptoms the lexicon reads, and 0.33–0.41 for the LLMs.
- **Caveat:** these transcripts come from the same generator as the training data (new cards, same style), so they are in-distribution for the fine-tuned models and not for the baselines.

**Common errors.** Both fine-tuned models confuse fungal skin infection and contact dermatitis with chickenpox. Medium also confuses BPPV with heat exhaustion, and small confuses strep throat with mononucleosis.

**Limitations.**
- **The rule-based patient favours the NB doctor.** It often misreads a natural question (5.2, 5.4), and the NB doctor's questions are built from the patient's own lexicon, so the GPT-2 doctor's evidence is under-counted more.
- **Repeats are under-counted.** Only word-for-word repeats are counted; near-repeats such as "Does your ear feel red or swollen?" / "Is your ear red or swollen?" are not.
- **The selected doctor may not be the better one.** The doctor in `checkpoints/doctor/best` is GPT-2 small: on 50 dev cases it led medium on top-3 by two cases (0.92 vs 0.88). Medium is better on the test set in both modes. Changing the choice because of test results would mean selecting on the test set; a larger dev set would settle it fairly.
